In [0]:
from delta.tables import *
from pyspark.sql import DataFrame

def create_table():

  spark.sql("create catalog if not exists patterns")
  spark.sql("create schema if not exists patterns.mastering")
  spark.sql("drop table if exists patterns.mastering.customer")

  spark.sql("""
    create table patterns.mastering.customer (
      quid long GENERATED ALWAYS AS IDENTITY,
      mastered_on_source_system_id int not null,
      mastered_on_id string not null,
      _job_run_id long not null,
      _created_date timestamp not null,
      _updated_date timestamp not null,
      CONSTRAINT customer_pk PRIMARY KEY(quid) RELY
    )
    CLUSTER BY (quid, mastered_on_id)""")
  
  
def create_source_data():
  spark.sql("create catalog if not exists patterns")
  spark.sql("create schema if not exists patterns.mastering")
  spark.sql("drop table if exists patterns.mastering.customer_source")
  spark.sql("""
    create table patterns.mastering.customer_source (
        compound_id long GENERATED ALWAYS AS IDENTITY,
        `id` long,
        id_1 string,
        id_2 string,
        id_3 string,
        id_4 string,
        id_5 string,
        ssid_1 int,
        ssid_2 int,
        ssid_3 int,
        ssid_4 int,
        ssid_5 int,
        _job_run_id long not null,
        CONSTRAINT customer_source_pk PRIMARY KEY(compound_id) RELY
      )
      CLUSTER BY (compound_id)
    """)
  
  spark.sql("""
    insert into patterns.mastering.customer_source
    (
        `id`,
        id_1,
        id_2,
        id_3,
        id_4,
        id_5,
        ssid_1,
        ssid_2,
        ssid_3,
        ssid_4,
        ssid_5,
        _job_run_id
    )
    select
      d.id,
      d.id_1, d.id_2, d.id_3, d.id_4, d.id_5,
      d.ssid_1, d.ssid_2, d.ssid_3, d.ssid_4, d.ssid_5,
      d._job_run_id
    from
    (
      select
        row_number() over(order by c.c_customer_sk, c.t_time_sk) as `id`,
        null as id_1  , 
        null as id_2  , 
        null as id_3  , 
        null as id_4  , 
        concat("5-", c.c_customer_sk) as id_5,
        null as ssid_1, null as ssid_2, null as ssid_3, null as ssid_4, 5 as ssid_5,
      1 as _job_run_id
      from (
        select c.c_customer_sk, t_time_sk from samples.tpcds_sf1.customer c
        join samples.tpcds_sf1.time_dim on t_time_sk < 10
      ) c

      union all

      select
        row_number() over(order by c.c_customer_sk, c.t_time_sk) as `id`,
        null as id_1  , 
        null as id_2  , 
        null as id_3  , 
        concat("4-", c.c_customer_sk) as id_4 , 
        concat("5-", c.c_customer_sk) as id_5,
        null as ssid_1, null as ssid_2, null as ssid_3, 4 as ssid_4, 5 as ssid_5,
      2 as _job_run_id
      from (
        select c.c_customer_sk, t_time_sk from samples.tpcds_sf1.customer c
        join samples.tpcds_sf1.time_dim on t_time_sk < 10
      ) c

      union all

      select
        row_number() over(order by c.c_customer_sk, c.t_time_sk) as `id`,
        null as id_1  , 
        null as id_2  , 
        concat("3-", c.c_customer_sk) as id_3 , 
        concat("4-", c.c_customer_sk) as id_4 , 
        concat("5-", c.c_customer_sk) as id_5,
        null as ssid_1, null as ssid_2, 3 as ssid_3, 4 as ssid_4, 5 as ssid_5,
      3 as _job_run_id
      from (
        select c.c_customer_sk, t_time_sk from samples.tpcds_sf1.customer c
        join samples.tpcds_sf1.time_dim on t_time_sk < 10
      ) c

      union all

      select
        row_number() over(order by c.c_customer_sk, c.t_time_sk) as `id`,
        null as id_1  , 
        concat("2-", c.c_customer_sk) as id_2 , 
        concat("3-", c.c_customer_sk) as id_3 , 
        concat("4-", c.c_customer_sk) as id_4 , 
        concat("5-", c.c_customer_sk) as id_5,
        null as ssid_1, 2 as ssid_2, 3 as ssid_3, 4 as ssid_4, 5 as ssid_5,
      4 as _job_run_id 
      from (
        select c.c_customer_sk, t_time_sk from samples.tpcds_sf1.customer c
        join samples.tpcds_sf1.time_dim on t_time_sk < 10
      ) c

      union all

      select
        row_number() over(order by c.c_customer_sk, c.t_time_sk) as `id`,
        concat("1-", c.c_customer_sk) as id_1 , 
        concat("2-", c.c_customer_sk) as id_2 , 
        concat("3-", c.c_customer_sk) as id_3 , 
        concat("4-", c.c_customer_sk) as id_4 , 
        concat("5-", c.c_customer_sk) as id_5,
        1 as ssid_1, 2 as ssid_2, 3 as ssid_3, 4 as ssid_4, 5 as ssid_5,
      5 as _job_run_id 
      from (
        select c.c_customer_sk, t_time_sk from samples.tpcds_sf1.customer c
        join samples.tpcds_sf1.time_dim on t_time_sk < 10
      ) c
    ) d
  """)

  spark.sql("""
    update patterns.mastering.customer_source
    set id_5 = concat(cast(ssid_5 as string), "-", cast(`id` as string))
    where ssid_5 = 5
  """)
  spark.sql("""
    update patterns.mastering.customer_source
    set id_4 = concat(cast(ssid_4 as string), "-", cast(`id` as string))
    where ssid_4 = 4
  """)  
  spark.sql("""
    update patterns.mastering.customer_source
    set id_3 = concat(cast(ssid_3 as string), "-", cast(`id` as string))
    where ssid_3 = 3
  """)  
  spark.sql("""
    update patterns.mastering.customer_source
    set id_2 = concat(cast(ssid_2 as string), "-", cast(`id` as string))
    where ssid_2 = 2
  """)  
  spark.sql("""
    update patterns.mastering.customer_source
    set id_1 = concat(cast(ssid_1 as string), "-", cast(`id` as string))
    where ssid_1 = 1
  """)
    
def get_source_data(job_run_id:int):
  df = spark.sql(f"""
    select
      compound_id,
      d.id_1, d.id_2, d.id_3, d.id_4, d.id_5,
      d.ssid_1, d.ssid_2, d.ssid_3, d.ssid_4, d.ssid_5,
      d._job_run_id, 
      current_date() as _created_date,
      current_date() as _updated_date
    from patterns.mastering.customer_source d
    where d.`_job_run_id` = {job_run_id}
  """, job_run_id=job_run_id)
  return df

def merge_data():

  df = spark.sql("select * from patterns.mastering.stage_customer_mastering")

  destination = DeltaTable.forName(spark, "patterns.mastering.customer")

  audit = (
    destination.alias('dst') 
    .merge(
      df.alias('src'), "src.quid = dst.quid"
    )
    .whenNotMatchedInsert(
      values={
        "mastered_on_source_system_id"  : "src.mastered_on_source_system_id",
        "mastered_on_id"                : "src.mastered_on_id",
        "_job_run_id"                   : "src._job_run_id",
        "_created_date"                 : "src._created_date",
        "_updated_date"                 : "src._updated_date"
      }
    )
    .whenMatchedUpdate(
      condition="src.mastered_on_id!=dst.mastered_on_id",
      set={
        "mastered_on_source_system_id"  : "src.mastered_on_source_system_id",
        "mastered_on_id"                : "src.mastered_on_id",
        "_job_run_id"                   : "src._job_run_id",
        "_updated_date"                 : "src._updated_date"
      }
    )
    .execute()
  )
  return audit

In [0]:
# def stage_data(job_run_id:int):

#   df = get_source_data(job_run_id)
#   df = spark.sql("""
#     create or replace temp table temp_customer_mastering as

#     select
#         d.quid,
#         coalesce(s.id_1, s.id_2, s.id_3, s.id_4, s.id_5)            as mastered_on_id,
#         coalesce(s.ssid_1, s.ssid_2, s.ssid_3, s.ssid_4, s.ssid_5)  as mastered_on_source_system_id,
#         d.mastered_on_id                                            as current_mastered_on_id,
#         d.mastered_on_source_system_id                              as current_mastered_on_source_system_id,
#         s._job_run_id, 
#         current_date()                                              as _created_date,
#         current_date()                                              as _updated_date
#     from {df} s
#     left join patterns.mastering.customer d
#     on (
#           d.mastered_on_id                = s.id_1
#       and d.mastered_on_source_system_id  = s.ssid_1
#     )
#     or
#     (
#           d.mastered_on_id                = s.id_2
#       and d.mastered_on_source_system_id  = s.ssid_2
#       and s.ssid_1 is null
#     )
#     or
#     (
#           d.mastered_on_id                = s.id_3
#       and d.mastered_on_source_system_id  = s.ssid_3
#       and s.ssid_1 is null
#       and s.ssid_2 is null
#     )
#     or
#     (
#           d.mastered_on_id                = s.id_4
#       and d.mastered_on_source_system_id  = s.ssid_4
#       and s.ssid_1 is null
#       and s.ssid_2 is null 
#       and s.ssid_3 is null
#     )
#     or
#     (
#           d.mastered_on_id                = s.id_5
#       and d.mastered_on_source_system_id  = s.ssid_5
#       and s.ssid_1 is null
#       and s.ssid_2 is null 
#       and s.ssid_3 is null
#       and s.ssid_4 is null
#     )

#   """, df=df)


In [0]:
def stage_data(df):

  spark.sql("create catalog if not exists patterns")
  spark.sql("create schema if not exists patterns.mastering")
  spark.sql("drop table if exists patterns.mastering.stage_customer_mastering")
  spark.sql("""
    create table patterns.mastering.stage_customer_mastering (
        quid long,
        compound_id long,
        mastered_on_id string,
        mastered_on_source_system_id int,
        current_mastered_on_id string,
        current_mastered_on_source_system_id int,
        _job_run_id long,
        _created_date date,
        _updated_date date,
        CONSTRAINT stage_customer_mastering_pk PRIMARY KEY(compound_id) RELY
      )
      CLUSTER BY (quid, compound_id)
    """)

  audit_1 = spark.sql("""
    merge into patterns.mastering.stage_customer_mastering as dst
    using (
      select
          d.quid,
          s.compound_id,
          coalesce(s.id_1, s.id_2, s.id_3, s.id_4, s.id_5)            as mastered_on_id,
          coalesce(s.ssid_1, s.ssid_2, s.ssid_3, s.ssid_4, s.ssid_5)  as mastered_on_source_system_id,
          d.mastered_on_id                                            as current_mastered_on_id,
          d.mastered_on_source_system_id                              as current_mastered_on_source_system_id,
          s._job_run_id, 
          current_date()                                              as _created_date,
          current_date()                                              as _updated_date
      from {df} s
      join patterns.mastering.customer d
      on d.mastered_on_id                   = s.id_1
        and d.mastered_on_source_system_id  = s.ssid_1
      where s.ssid_1 is not null
    ) src on src.compound_id = dst.compound_id
    when not matched then insert *
  """, df=df)

  print("step 1: finished")

  audit_2 = spark.sql("""
    merge into patterns.mastering.stage_customer_mastering as dst
    using (
      select
          d.quid,
          s.compound_id,
          coalesce(s.id_1, s.id_2, s.id_3, s.id_4, s.id_5)            as mastered_on_id,
          coalesce(s.ssid_1, s.ssid_2, s.ssid_3, s.ssid_4, s.ssid_5)  as mastered_on_source_system_id,
          d.mastered_on_id                                            as current_mastered_on_id,
          d.mastered_on_source_system_id                              as current_mastered_on_source_system_id,
          s._job_run_id, 
          current_date()                                              as _created_date,
          current_date()                                              as _updated_date
      from {df} s
      join patterns.mastering.customer d
      on d.mastered_on_id                   = s.id_2
        and d.mastered_on_source_system_id  = s.ssid_2
      where s.ssid_2 is not null
    ) src on src.compound_id = dst.compound_id
    when not matched then insert *
  """, df=df)

  print("step 2: finished")

  audit_3 = spark.sql("""
    merge into patterns.mastering.stage_customer_mastering as dst
    using (
      select
          d.quid,
          s.compound_id,
          coalesce(s.id_1, s.id_2, s.id_3, s.id_4, s.id_5)            as mastered_on_id,
          coalesce(s.ssid_1, s.ssid_2, s.ssid_3, s.ssid_4, s.ssid_5)  as mastered_on_source_system_id,
          d.mastered_on_id                                            as current_mastered_on_id,
          d.mastered_on_source_system_id                              as current_mastered_on_source_system_id,
          s._job_run_id, 
          current_date()                                              as _created_date,
          current_date()                                              as _updated_date
      from {df} s
      join patterns.mastering.customer d
      on d.mastered_on_id                   = s.id_3
        and d.mastered_on_source_system_id  = s.ssid_3
      where s.ssid_3 is not null
    ) src on src.compound_id = dst.compound_id
    when not matched then insert *
  """, df=df)

  print("step 3: finished")

  audit_4 = spark.sql("""
    merge into patterns.mastering.stage_customer_mastering as dst
    using (
      select
          d.quid,
          s.compound_id,
          coalesce(s.id_1, s.id_2, s.id_3, s.id_4, s.id_5)            as mastered_on_id,
          coalesce(s.ssid_1, s.ssid_2, s.ssid_3, s.ssid_4, s.ssid_5)  as mastered_on_source_system_id,
          d.mastered_on_id                                            as current_mastered_on_id,
          d.mastered_on_source_system_id                              as current_mastered_on_source_system_id,
          s._job_run_id, 
          current_date()                                              as _created_date,
          current_date()                                              as _updated_date
      from {df} s
      join patterns.mastering.customer d
      on d.mastered_on_id                   = s.id_4
        and d.mastered_on_source_system_id  = s.ssid_4
      where s.ssid_4 is not null
    ) src on src.compound_id = dst.compound_id
    when not matched then insert *
  """, df=df)

  print("step 4: finished")

  audit_5 = spark.sql("""
    merge into patterns.mastering.stage_customer_mastering as dst
    using (
      select
          d.quid,
          s.compound_id,
          coalesce(s.id_1, s.id_2, s.id_3, s.id_4, s.id_5)            as mastered_on_id,
          coalesce(s.ssid_1, s.ssid_2, s.ssid_3, s.ssid_4, s.ssid_5)  as mastered_on_source_system_id,
          d.mastered_on_id                                            as current_mastered_on_id,
          d.mastered_on_source_system_id                              as current_mastered_on_source_system_id,
          s._job_run_id, 
          current_date()                                              as _created_date,
          current_date()                                              as _updated_date
      from {df} s
      join patterns.mastering.customer d
      on d.mastered_on_id                   = s.id_5
        and d.mastered_on_source_system_id  = s.ssid_5
      where s.ssid_5 is not null
    ) src on src.compound_id = dst.compound_id
    when not matched then insert *
  """, df=df)

  print("step 5: finished")

  audit_6 = spark.sql("""
    merge into patterns.mastering.stage_customer_mastering as dst
    using (
      select
          -1 as quid,
          s.compound_id,
          coalesce(s.id_1, s.id_2, s.id_3, s.id_4, s.id_5)            as mastered_on_id,
          coalesce(s.ssid_1, s.ssid_2, s.ssid_3, s.ssid_4, s.ssid_5)  as mastered_on_source_system_id,
          null                                                        as current_mastered_on_id,
          null                                                        as current_mastered_on_source_system_id,
          s._job_run_id, 
          current_date()                                              as _created_date,
          current_date()                                              as _updated_date
      from {df} s
    ) src on src.compound_id = dst.compound_id
    when not matched then insert *
  """, df=df)

  print("step 6: finished")

  return audit_1.union(audit_2).union(audit_3).union(audit_4).union(audit_5).union(audit_6)


In [0]:
create_table()
create_source_data()

In [0]:
df = get_source_data(3)
stage_data(df).display()
merge_data().display()

In [0]:
%sql
select * from patterns.mastering.stage_customer_mastering order by compound_id

In [0]:
%sql
select * from patterns.mastering.customer

In [0]:
%sql

select * from patterns.mastering.customer_source
where id = 2
ORDER BY _job_run_id
